# Duka Electronics: Half-Year Sales Exploration

**Analyst:** EUNICE LIVINGPROOF | **Cohort:** 1MILDEVS

Data: `duka_sales.csv` (1,000 orders, January to June 2026, generator seed 2026).

## Executive Summary

- Nairobi is the strongest branch, earning KES 26,106,100 (36.4% of the KES 71,792,500 half-year revenue), 1.9 times the revenue of Mombasa.
- Laptop is the top product by revenue at KES 41,535,000 (57.9% of revenue) while ranking number 2 for units sold (639 units), because each one costs KES 65,000.
- Weekend orders average KES 71,972 against KES 71,724 on weekdays (+0.3%), and 27.7% of all 1,000 orders fall on weekends.
- April 2026 was the best month at KES 13,518,300 and June 2026 the weakest at KES 10,903,400, a gap of 24.0%.
- Online orders number 320 (32.0% of orders) with an average of KES 71,207, which is +0.6% against KES 70,759 in store.

In [1]:
import os

import numpy as np
import openpyxl
import pandas as pd

os.makedirs("outputs", exist_ok=True)

## 1. First-five-minutes inspection

In [2]:
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Shape:", sales.shape)
display(sales.head())
display(sales.tail())

Shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


In [3]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB


In [4]:
display(sales.describe())
print("Missing values per column:")
print(sales.isna().sum())
print("Date range:", sales["order_date"].min().date(), "to", sales["order_date"].max().date())

,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64
Date range: 2026-01-01 to 2026-06-29


In [5]:
for col in ["branch", "product", "channel"]:
    display(sales[col].value_counts(normalize=True).round(3))

branch
Nairobi    0.376
Mombasa    0.186
Nakuru     0.162
Kisumu     0.159
Eldoret    0.117
Name: proportion, dtype: float64

product
Laptop        0.220
Tablet        0.215
Phone         0.194
Headphones    0.187
Charger       0.184
Name: proportion, dtype: float64

channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64

### Inspection notes

- **Shape:** 1,000 rows and 7 columns (order_id, order_date, branch, product, quantity, channel, unit_price).
- **Types:** `order_date` is a real datetime because of `parse_dates`; quantity and unit_price are integers; branch, product and channel are text.
- **Missing values:** 0 across the whole file.
- **Duplicate order_ids:** 0.
- **Date range:** 2026-01-01 to 2026-06-29 (179 distinct trading days).
- **Key categories:** 5 branches, 5 products and 3 channels. Channel mix: In-store 57.9%, Online 32.0%, Phone order 10.1%.

## 2. Derived columns

`revenue` is quantity times unit price. `month`, `weekday` and `is_weekend` come from
the `.dt` accessor, and `price_band` uses `np.where` (Premium is KES 28,000 or more).

In [6]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

display(sales.head())
for col in ["month", "weekday", "is_weekend", "price_band"]:
    display(sales[col].value_counts())

/tmp/ipykernel_2635/2163549121.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


month
March       186
January     173
April       167
June        162
February    159
May         153
Name: count, dtype: int64

weekday
Friday       166
Sunday       145
Thursday     145
Monday       140
Wednesday    137
Tuesday      135
Saturday     132
Name: count, dtype: int64

is_weekend
False    723
True     277
Name: count, dtype: int64

price_band
Value      565
Premium    435
Name: count, dtype: int64

## 3. Branch ranking: revenue versus number of orders

In [7]:
by_branch = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
)
by_branch["revenue_rank"] = by_branch["total_revenue"].rank(ascending=False, method="min").astype(int)
by_branch["orders_rank"] = by_branch["orders"].rank(ascending=False, method="min").astype(int)
by_branch["revenue_share_pct"] = (by_branch["total_revenue"] / by_branch["total_revenue"].sum() * 100).round(1)
by_branch = by_branch.sort_values("total_revenue", ascending=False)
display(by_branch)

,total_revenue,orders,revenue_rank,orders_rank,revenue_share_pct
branch,,,,,
Nairobi,26106100,376,1,1,36.4
Mombasa,14003700,186,2,2,19.5
Nakuru,11970400,162,3,3,16.7
Kisumu,10868100,159,4,4,15.1
Eldoret,8844200,117,5,5,12.3


### Answer

**The rankings are the same.** By revenue and by orders the order is Nairobi, Mombasa, Nakuru, Kisumu, Eldoret. Products are spread across the branches in much the same way, so a branch with more orders also earns proportionally more revenue. Nairobi leads with 36.4% of revenue (KES 26,106,100), 1.9 times Mombasa.

## 4. Top three products: revenue versus units

In [8]:
by_product = sales.groupby("product").agg(
    revenue=("revenue", "sum"),
    units=("quantity", "sum"),
    unit_price=("unit_price", "first"),
)
print("Top 3 by revenue")
display(by_product["revenue"].nlargest(3))
print("Top 3 by units sold")
display(by_product["units"].nlargest(3))
display(by_product.sort_values("revenue", ascending=False))

Top 3 by revenue


product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64

Top 3 by units sold


product
Tablet    691
Laptop    639
Phone     592
Name: units, dtype: int64

,revenue,units,unit_price
product,,,
Laptop,41535000,639,65000
Tablet,19348000,691,28000
Phone,8880000,592,15000
Headphones,1387500,555,2500
Charger,642000,535,1200


### Answer

**By revenue** the top three are Laptop, Tablet, Phone. **By units sold** they are Tablet, Laptop, Phone. Revenue is quantity times unit price, so price matters as much as volume. The same three products appear in both lists. Laptop generates 57.9% of all revenue.

## 5. Weekend versus weekday orders

In [9]:
weekend_table = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean"),
    trading_days=("order_date", "nunique"),
)
weekend_table["revenue_per_day"] = weekend_table["total_revenue"] / weekend_table["trading_days"]
weekend_table = weekend_table.rename(index={False: "Weekday", True: "Weekend"}).round(0)
display(weekend_table)

,orders,total_revenue,avg_order_revenue,trading_days,revenue_per_day
is_weekend,,,,,
Weekday,723,51856200,71724.0,128,405127.0
Weekend,277,19936300,71972.0,51,390908.0


### Answer

- **Weekdays:** 723 orders, KES 51,856,200 total revenue, KES 71,724 average per order.
- **Weekends:** 277 orders (27.7% of all orders), KES 19,936,300 total revenue, KES 71,972 average per order.
- The average weekend order is 0.3% higher than the average weekday order.
- Weekends are only 2 days in 7, so total revenue is misleading. Per trading day, weekends bring in KES 390,908 against KES 405,127 on weekdays, which is 3.5% lower.

## 6. Best and worst month

In [10]:
monthly = sales.groupby(sales["order_date"].dt.to_period("M"))["revenue"].sum()
display(monthly)

best_month, worst_month = monthly.idxmax(), monthly.idxmin()
gap_pct = (monthly.max() - monthly.min()) / monthly.min() * 100
print(f"Best month : {best_month} (KES {monthly.max():,})")
print(f"Worst month: {worst_month} (KES {monthly.min():,})")
print(f"Best month is {gap_pct:.1f}% higher than the worst month")

order_date
2026-01    11786600
2026-02    11772000
2026-03    12396800
2026-04    13518300
2026-05    11415400
2026-06    10903400
Freq: M, Name: revenue, dtype: int64

Best month : 2026-04 (KES 13,518,300)
Worst month: 2026-06 (KES 10,903,400)
Best month is 24.0% higher than the worst month


### Answer

**April 2026** is the best month with KES 13,518,300, and **June 2026** is the worst with KES 10,903,400. The best month earned 24.0% more than the worst. Caution: the worst month, June, has orders on only 28 of its 30 days in this extract, so the comparison is slightly unfair to it.

## 7. Premium online orders from Nairobi and Mombasa

In [11]:
mask = (
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
)
premium_online = sales.loc[mask]
print("Orders:", len(premium_online))
print(f"Total revenue: KES {premium_online['revenue'].sum():,}")
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)

check = pd.read_csv("outputs/premium_online_coast_capital.csv")
print("Export check ->", check["price_band"].unique(), check["channel"].unique(), sorted(check["branch"].unique()))

Orders: 83
Total revenue: KES 10,218,000
Export check -> <StringArray>
['Premium']
Length: 1, dtype: str <StringArray>
['Online']
Length: 1, dtype: str ['Mombasa', 'Nairobi']


### Answer

There are **83** Premium online orders from Nairobi and Mombasa, with total revenue of **KES 10,218,000**, which is 14.2% of all revenue. Nairobi accounts for KES 6,522,000 of that and Mombasa for KES 3,696,000. The orders are saved in `outputs/premium_online_coast_capital.csv` without the index.

## 8. Spreadsheet task: Excel totals versus pandas

`excel/duka_branch_summary.xlsx` contains a `SUMIFS` branch revenue table (Summary sheet),
an `XLOOKUP` region column (Sales sheet) and a PivotTable of revenue by branch and
channel (Pivot sheet). The cell below reads the totals Excel stored and compares them
with pandas.

In [12]:
wb = openpyxl.load_workbook("excel/duka_branch_summary.xlsx", data_only=True)

ws = wb["Summary"]
excel_totals = {ws.cell(row=r, column=1).value: ws.cell(row=r, column=2).value for r in range(2, 7)}
assert all(v is not None for v in excel_totals.values()), (
    "No stored values: open the workbook in Excel, save it, and copy it back (Phase 4)."
)

pandas_totals = sales.groupby("branch")["revenue"].sum()
comparison = pd.DataFrame({
    "pandas": pandas_totals,
    "excel_sumifs": pd.Series(excel_totals),
})
comparison["match"] = comparison["pandas"] == comparison["excel_sumifs"]
display(comparison)
assert comparison["match"].all(), "Excel SUMIFS totals do not match pandas"

# PivotTable: find the Grand Total row and compare with total revenue
assert "Pivot" in wb.sheetnames, "Add the PivotTable on a sheet named 'Pivot' (Phase 4)."
grand_total = None
for row in wb["Pivot"].iter_rows(values_only=True):
    if any(isinstance(c, str) and c.strip().lower() == "grand total" for c in row):
        grand_total = [c for c in row if isinstance(c, (int, float))][-1]
print("PivotTable grand total:", grand_total, "| pandas total:", sales["revenue"].sum())
assert grand_total == sales["revenue"].sum(), "PivotTable total does not match pandas"
print("Excel SUMIFS and PivotTable totals match pandas.")

,pandas,excel_sumifs,match
Eldoret,8844200,8844200,True
Kisumu,10868100,10868100,True
Mombasa,14003700,14003700,True
Nairobi,26106100,26106100,True
Nakuru,11970400,11970400,True


PivotTable grand total: 71792500 | pandas total: 71792500
Excel SUMIFS and PivotTable totals match pandas.
